# VM HQ — Can 두 질문: `can_tent12i_hq` × 3 + `can_prefill_t12i` × 3 (vm2_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로** 실행한다. 1번은 런타임을 재시작하므로 재시작 뒤 2번부터. 코드 변경 없음, 6 run(RAM ≈ 102 GB, 5~6시간), 마지막 프로세스가 끝나면 9번이 VM을 반납한다. 기준 문서: `results/2026-09-08/README.md`, `HANDOFF.md` §21.

**Square hq(9/7, 9/8 재분석)**: tent12 대비 127k +0.093 ± 0.020(3/3), Q_W 23,000 → −76이면서 엔트로피 12 유지 → bonus 가설 **지지**. 초기 AUC −0.050 ± 0.040, 42k 0.33 < π_dp 0.494 → dip은 남음. 여기서는 그 다음 두 질문만 돌린다.

## 질문 1 — `can_tent12i_hq_s{1,2,3}`: 엔트로피를 붙든 상태에서 β=0(hard target)이 Can에서 무해한가
`train.ent_coef=auto_0.3 train.target_ent=12 train.critic_entropy_scale=0.0`, 리플레이·사전학습 없음. 상호작용 가설 명시: `can_hardq`(β=0, 목표 0)는 AUC 0.298로 baseline 0.409보다 나빴다 → "β=0이 좋다"가 아니라 "엔트로피를 붙든 상태에서만 β=0이 무해/이득".
- **무해**: 평균곡선 최저 ≥ 0.405(π_dp; tent12i 0.460) **그리고** 129k ≥ 0.53(baseline n=5 0.532) → "엔트로피 유지 + hard target"을 두 과제 공통 처방으로.
- **해로움**: 어느 하나 미달 → hard target의 초반 비용이 실재(Square hq의 −0.05가 잡음이 아님) → 그때만 β 스케줄(dip 창 soft → hard) 구현.
- 진단: `logp_mean` ≈ −12, `qw_mean`이 t12i(+100대)와 달리 실제 수익 눈금(음수). 비교군 `can_tent12i`(AUC 0.594 ± 0.052, 129k 0.697 ± 0.058), `can_fixalpha_03`, `can_baseline`, `can_hardq`.

## 질문 2 — `can_prefill_t12i_s{1,2,3}`: 데모 prefill + 엔트로피 유지를 합치면 둘 다 얻는가
`offline_mix.mode=prefill offline_data_path=… train.ent_coef=auto_0.3 train.target_ent=12`(`can_mix_prefill`과 같은 prefill, `can_tent12i`와 같은 온도). prefill 단독 129k 0.858 ± 0.049·순간 최저 0.093; t12i 단독 최저 0.460·129k 0.697. 엔트로피 설명이 맞으면 prefill의 깊은 dip도 auto-α(목표 0) 붕괴이므로 t12i가 없애야 한다.
- **둘 다**: 최저 ≥ 0.405 **그리고** 129k ≥ 0.80 → Can 처방 완성.
- **부분**: 최저 ≥ 0.405, 129k < 0.80 → 데모가 있어도 엔트로피 유지의 후반 비용(Square형 상충).
- **실패**: 최저 < 0.405 → prefill의 dip은 엔트로피 붕괴가 아님 → 엔트로피 설명의 경계.
- 비교군 `can_mix_prefill`, `can_tent12i`, `can_calql_prefill`(최저 0.360, 129k 0.765).

10번(선택) `square_tent12_hq_s{4,5}`는 n=5 정렬용이며 초기 약점을 없애 주지 않는다(README §9). 돌리려면 5b → 10.

## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Can 정책·정규화 복원과 환경 패치

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Can policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. 사전검사 — 데모 npz, 이번 exp_id 6개가 비어 있는지(있으면 같은 명령이 checkpoint resume), RAM·디스크

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python - <<'PY'
import numpy as np
path = '/content/drive/MyDrive/dsrl_project/offline/can_train_offline.npz'
with np.load(path) as data:
    assert len(data['states']) > 0
    print('OK', path.split('/')[-1], 'rows', len(data['states']), 'keys', sorted(data.files))
PY
for E in can_tent12i_hq_s1 can_tent12i_hq_s2 can_tent12i_hq_s3 can_prefill_t12i_s1 can_prefill_t12i_s2 can_prefill_t12i_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80) (같은 명령이면 resume)"; else echo "$E: 새로 시작"; fi
done
for G in tent12i mix_prefill fixalpha_03 hardq baseline; do echo -n "비교군 can_$G: "; ls -d $PROJ/logs/can_${G}_s* 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1

## 7. 6개 시작 — `can_tent12i_hq` × 3 + `can_prefill_t12i` × 3 (150k, 5k 격자). 질문 1만 돌리려면 두 번째 launch 줄을 지운다

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch can_tent12i_hq_s$S  seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False train.ent_coef=auto_0.3 train.target_ent=12 train.critic_entropy_scale=0.0
  launch can_prefill_t12i_s$S seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=prefill offline_data_path=$PROJ/offline/can_train_offline.npz train.ent_coef=auto_0.3 train.target_ent=12
done

## 8. 3분 후 자동 확인 — 6개가 running, ERR 없음, 프로세스 인자에 `critic_entropy_scale=0.0` / `offline_mix.mode=prefill`이 보여야 함. 20~30분 뒤 다시 실행하면 train_log의 α·logp·offline_p도 찍힌다

In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only can_tent12i_hq_s,can_prefill_t12i_s
for E in can_tent12i_hq_s1 can_tent12i_hq_s2 can_tent12i_hq_s3 can_prefill_t12i_s1 can_prefill_t12i_s2 can_prefill_t12i_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s logp_mean=%s qw_mean=%s offline_p=%s\n", $c["env_steps"], $c["ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["offline_p"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2

## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'can_{kind}_s{s}' for kind in ('tent12i_hq', 'prefill_t12i') for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

while True:
    procs = running()
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {last_event(e)}' for e in EXPECTED), flush=True)
    if not procs:
        print('all VM HQ runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)

## 5b. (선택) Square 정책·정규화 복원 — 10번을 돌릴 때만

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
NORM_REL=robomimic/square/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Square policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 10. (선택) `square_tent12_hq_s{4,5}` — n=5 정렬용. 띄운 뒤 9번 keepalive의 EXPECTED에 두 이름을 추가해 다시 실행

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
CFG='--config-path=cfg/robomimic --config-name=dsrl_square.yaml'
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 4 5; do
  launch square_tent12_hq_s$S seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False train.target_ent=12 train.critic_entropy_scale=0.0
done

## 11. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서 `~/Downloads/logs/`에 풀고
`.venv/bin/python scripts/plot_results.py --logs ~/Downloads/logs --out ~/Downloads/dsrl_figs_hq --axes "hq=baseline,tent12i,fixalpha_03,hardq,tent12i_hq;prefill_t12i=baseline,mix_prefill,tent12i,calql_prefill,prefill_t12i;square_hq=square_baseline,square_tent12,square_tent12_hq"`

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip